# FCVG101 Coursework – Vision Inspection Assistant

This notebook only calls functions in `src/` and shows their results (layer 3, see `PROJECT_GUIDE.md`). Every heading
carries the step IDs it runs, so you can search for an ID such as `[1B.3]` and find the code, this notebook and the report.

**Task 1 – Classical computer vision engine (20 pts)**

| Part | Rubric | What it shows | Code |
| --- | --- | --- | --- |
| 1A Camera calibration | 1.1 | The camera model: intrinsics K, lens distortion, undistortion | `src/calibration.py` |
| 1B Classical processing | 1.2, 1.3 | Spatial and frequency filtering, gradients and edges, interest points and matching | `src/filtering.py`, `frequency.py`, `edges.py`, `features.py` |

Each section follows the same pattern: **goal → theory → run → result → how to read it → conclusion**. The
conclusions marked *(computed)* are printed from the numbers of this run, so they stay correct when the images change.

## 0 Setup

### Running on Google Colab (optional, for a GPU)
1. Upload `FCVG101-Coursework.zip` to the top of your Google Drive (*My Drive*).
2. In Colab: **File → Upload notebook** → `notebooks/FCVG101_Coursework.ipynb`, then **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all.** The cell below mounts Drive, unzips the project to `/content`, installs `timm` and downloads
   the flowers dataset on first use. Task 2 retrains on the GPU (≈10–15 min instead of ≈55 min on a laptop CPU).
   Latency/size are still measured on the CPU, because the Task 3 app runs on a CPU.
4. The last cell copies the new `results/` and `artifacts/` back to Drive as `FCVG101-outputs.zip`
   (everything in `/content` is deleted when the runtime stops).

On a local machine this cell does nothing.

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir('/content/FCVG101-Coursework'):
        subprocess.run(['unzip', '-q', '/content/drive/MyDrive/FCVG101-Coursework.zip', '-d', '/content'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
    os.chdir('/content/FCVG101-Coursework')

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from src.config import load_config, set_seed
from src.io_utils import list_images, read_image
from src.visualize import show_grid

cfg = load_config()
set_seed(cfg.seed)
pd.set_option('display.precision', 4)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import os
    cfg.task2.num_workers = min(8, os.cpu_count())   # Colab is Linux: parallel image loading keeps the GPU busy
print('Project root:', cfg.project_root)
print('OpenCV', cv2.__version__)
import torch
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

## 1A Camera calibration — [1A.1] … [1A.6]

### Goal
Estimate how the camera turns 3D points into pixels, so that (a) lens distortion can be removed and (b) later tasks
can relate pixels to real geometry. Rubric 1.1 asks for the intrinsic matrix, the distortion coefficients, a
quality measure and a corrected image.

### Theory
**Pinhole model.** A 3D point $X$ on the board maps to pixel $x$ by
$$ s\,\begin{bmatrix}u\\v\\1\end{bmatrix} = K\,[R\,|\,t]\begin{bmatrix}X\\Y\\Z\\1\end{bmatrix},\qquad
K=\begin{bmatrix}f_x & 0 & c_x\\ 0 & f_y & c_y\\ 0&0&1\end{bmatrix}$$
- $f_x, f_y$: focal length in pixels (focal length in mm ÷ pixel size). Nearly equal for square pixels.
- $c_x, c_y$: principal point, where the optical axis hits the sensor; usually near the image centre.
- $R, t$ (*extrinsics*): where the board was for each photo. They differ per view; $K$ is shared by all views.

**Lens distortion.** Real lenses bend straight lines. OpenCV models it with 5 coefficients
$[k_1, k_2, p_1, p_2, k_3]$: radial $x_d = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$ and tangential $p_1, p_2$ (lens not
parallel to the sensor). $k_1 < 0$ means **barrel** distortion (lines bow outwards, typical for wide-angle phone
cameras), $k_1 > 0$ **pincushion**.

**Zhang's method.** A checkerboard gives points whose 3D positions are known exactly (on the plane $Z=0$, spaced by the
square size). Each photo gives the matching 2D pixels. From ≥ 3 views of the plane at different angles,
`cv2.calibrateCamera` solves for $K$, the distortion and every view's $R, t$, minimising the **reprojection error**:
the pixel distance between the detected corners and the corners re-projected through the fitted model.

### Inputs and outputs
| | |
| --- | --- |
| Input | 10–20 photos of the printed board in `data/calibration/`, **one camera, one resolution, fixed zoom/focus**; board size in `cfg.task1a` (INNER corners and measured square size) |
| Output | `artifacts/calibration.npz` (K, dist, width, height – read by Task 3 and `undistort_view_task1a.py`); `results/task1a/` corner overlays, `detection_log.csv`, `view_errors.csv`, `calibration_variants.csv`, `view_errors.png`, `before_after.jpg`, `metrics.json` |

### Pipeline
`[1A.1] load images` → `[1A.2a] 3D board points` → `[1A.2] detect corners` → `[1A.3] calibrate (baseline and fix_k3)` →
`[1A.4] reprojection error per view` → `[1A.5] undistort` → `[1A.6] save calibration`.

**Capture rules** (checked by `capture_checker_task1a.py`): sharp photos, the whole board visible, board tilted up to
about 45° in different directions, placed near the centre *and* the corners/edges of the frame, at different
distances; print at 100 % (no "fit to page") and measure a square with a ruler.

In [ ]:
# Which calibration set to use. Your real photos always win; the course pack's SYNTHETIC boards are only a stand-in
# so the pipeline can be shown before the photos exist.
cfg_1a = load_config()          # own copy, so the fallback below never changes the settings Task 1B uses
real_photos = list_images(cfg_1a.paths.calibration_images) if cfg_1a.paths.calibration_images.is_dir() else []
USING_SYNTHETIC = len(real_photos) == 0
if USING_SYNTHETIC:
    fallback = cfg_1a.task1a.synthetic_fallback
    cfg_1a.paths.calibration_images = cfg_1a.project_root / fallback.folder
    cfg_1a.task1a.pattern_cols, cfg_1a.task1a.pattern_rows = fallback.pattern_cols, fallback.pattern_rows
    cfg_1a.task1a.square_size_mm = fallback.square_size_mm
    # keep synthetic outputs apart: they must never become artifacts/calibration.npz used by the app
    cfg_1a.paths.results = cfg_1a.paths.results / 'synthetic_preview'
    cfg_1a.paths.artifacts = cfg_1a.paths.results / 'artifacts'
    display(Markdown('> **SYNTHETIC preview.** `data/calibration/` is empty, so 1A runs on the course pack\'s rendered '
                     'boards (known camera). These numbers show the method works; they are **not** a calibration of '
                     'your camera and must not go in the report as one.'))
print('Calibration images:', cfg_1a.paths.calibration_images)
print('Outputs:', cfg_1a.paths.results / 'task1a', '|', cfg_1a.paths.artifacts / 'calibration.npz')
print(f'Board: {cfg_1a.task1a.pattern_cols} x {cfg_1a.task1a.pattern_rows} inner corners, '
      f'{cfg_1a.task1a.square_size_mm} mm squares')

### Run [1A.R]

In [ ]:
from src.calibration import run_task1a

report_1a = run_task1a(cfg_1a)
out_1a = cfg_1a.paths.results / 'task1a'
print('Results written to', out_1a)

### [1A.1]–[1A.2] Corner detection
`cv2.findChessboardCornersSB` (sector-based detector, sub-pixel accurate) is tried first; if it fails,
`cv2.findChessboardCorners` + `cv2.cornerSubPix` refines the corners to about 0.1 px. A photo is **rejected** when the
complete grid of inner corners is not found (board cut off, blur, glare, or wrong `pattern_cols/rows`).
The coloured overlays below are the evidence that the corners are on the right places and in the right order.

In [ ]:
display(pd.read_csv(out_1a / 'detection_log.csv'))
overlays = sorted((out_1a / 'corners').glob('*_corners.jpg'))
picks = [overlays[i] for i in np.linspace(0, len(overlays) - 1, min(3, len(overlays))).astype(int)]
fig = show_grid([read_image(p) for p in picks], [p.stem for p in picks], cols=3)
plt.show()

In [ ]:
# Conclusion (computed)
n_ok, n_all = len(report_1a.accepted), len(report_1a.accepted) + len(report_1a.rejected)
text = f'**Conclusion.** {n_ok} of {n_all} photos gave a complete corner grid and were used. '
text += ('Every photo was usable. ' if not report_1a.rejected else
         f'Rejected: {", ".join(n for n, _ in report_1a.rejected)} – check them for blur or a cut-off board. ')
text += ('This is inside the recommended 10–20 views.' if 10 <= n_ok <= 20 else
         'The guide recommends 10–20 varied views; take more photos.' if n_ok < 10 else
         'More than 20 views is fine but adds little.')
display(Markdown(text))

### [1A.3] Intrinsics and distortion: baseline vs `fix_k3`
- **Baseline**: all 5 distortion coefficients are free.
- **Advanced (`fix_k3`)**: $k_3$ is held at 0. With a normal lens and views that do not reach far into the image
  corners, $k_3$ is poorly constrained: the solver can push $k_2$ and $k_3$ to large values of opposite sign that cancel
  each other on the data but extrapolate badly at the borders. If fixing $k_3$ barely changes the RMS, the simpler
  model is the more trustworthy one.

**How to read the table:** `rms_px` is the reprojection error OpenCV minimises (below 0.5 px is good for phone photos,
below 1 px acceptable); `fx, fy, cx, cy` in pixels; `k1 … k3` are unit-less.

In [ ]:
variants = pd.DataFrame(report_1a.variants).T
display(variants)
K = report_1a.calibration.K
print('Chosen variant:', 'fix_k3' if cfg_1a.task1a.fix_k3 else 'baseline')
print('K =\n', K.round(2))
print('dist [k1 k2 p1 p2 k3] =', report_1a.calibration.dist.ravel().round(5))

In [ ]:
# Conclusion (computed)
w, h = report_1a.calibration.image_size
base, fixed = report_1a.variants['baseline'], report_1a.variants['fix_k3']
lines = [f"- Focal length ≈ {base['fx']:.0f} × {base['fy']:.0f} px; fx/fy = {base['fx'] / base['fy']:.3f} "
         f"({'square pixels, as expected' if abs(base['fx'] / base['fy'] - 1) < 0.02 else 'pixels not square – check the views'}).",
         f"- Principal point ({base['cx']:.0f}, {base['cy']:.0f}) vs image centre ({w / 2:.0f}, {h / 2:.0f}): offset "
         f"{np.hypot(base['cx'] - w / 2, base['cy'] - h / 2):.0f} px "
         f"({np.hypot(base['cx'] - w / 2, base['cy'] - h / 2) / np.hypot(w, h) * 100:.1f} % of the diagonal).",
         f"- k1 = {base['k1']:.3f}: {'barrel' if base['k1'] < 0 else 'pincushion'} distortion.",
         f"- Baseline RMS {base['rms_px']:.4f} px vs fix_k3 RMS {fixed['rms_px']:.4f} px "
         f"(+{fixed['rms_px'] - base['rms_px']:.4f} px). Baseline k2 = {base['k2']:.3f}, k3 = {base['k3']:.3f}."]
if abs(base['k3']) > 0.5 and fixed['rms_px'] - base['rms_px'] < 0.01:
    lines.append('- **The baseline overfits**: a large k3 buys almost no error reduction. Prefer `fix_k3` '
                 '(set `task1a.fix_k3: true` in config).')
else:
    lines.append('- k3 stays small, so the 5-coefficient baseline is not overfitting here.')
if USING_SYNTHETIC:
    fb = cfg_1a.task1a.synthetic_fallback
    true_K, true_d = np.array(fb.true_K), np.array(fb.true_dist)
    lines.append(f"- **Ground-truth check (synthetic only):** true fx, fy, cx, cy = {true_K[0, 0]:.0f}, {true_K[1, 1]:.0f}, "
                 f"{true_K[0, 2]:.0f}, {true_K[1, 2]:.0f}; fix_k3 estimate = {fixed['fx']:.1f}, {fixed['fy']:.1f}, "
                 f"{fixed['cx']:.1f}, {fixed['cy']:.1f} (focal error {abs(fixed['fx'] - true_K[0, 0]) / true_K[0, 0] * 100:.1f} %). "
                 f"True k1 = {true_d[0]}, estimated {fixed['k1']:.3f}: the method recovers the known camera.")
display(Markdown('**Conclusion.**\n' + '\n'.join(lines)))

### [1A.4] Reprojection error per view
For every corner: distance between where it was detected and where the fitted model puts it. One bar per photo;
the dashed line is the overall mean. A view far above the line usually has blur, a mis-detected corner or a
board that moved/bent – re-take or drop it and calibrate again.

In [ ]:
display(pd.read_csv(out_1a / 'view_errors.csv'))
display(Image(out_1a / 'view_errors.png'))

In [ ]:
# Conclusion (computed)
errors = pd.read_csv(out_1a / 'view_errors.csv')
mean_px = errors['mean_reprojection_error_px'].mean()
worst = errors.loc[errors['mean_reprojection_error_px'].idxmax()]
outliers = errors[errors['mean_reprojection_error_px'] > 2 * mean_px]['file_name'].tolist()
display(Markdown(
    f"**Conclusion.** Mean reprojection error {report_1a.mean_error_px:.3f} px over {len(errors)} views; worst view "
    f"`{worst['file_name']}` at {worst['mean_reprojection_error_px']:.3f} px. "
    + (f"Views above twice the mean: {', '.join(outliers)} – inspect their overlays. " if outliers else
       'No view is above twice the mean, so no single photo dominates the fit. ')
    + ('Sub-pixel error means the model explains the corners well. ' if report_1a.mean_error_px < 1 else
       'Error above 1 px: check focus, board flatness and the square size. ')
    + 'A low error alone does not prove a good calibration: the views must also be varied in angle and position, '
      'and straight lines must look straight after undistortion (next step).'))

### [1A.5] Undistortion and [1A.6] saving
`cv2.getOptimalNewCameraMatrix(K, dist, size, alpha)` then `cv2.undistort` re-maps every pixel through the inverse
distortion. `alpha = 0` (config) crops to pixels that are valid everywhere, `alpha = 1` keeps all source pixels and
shows black curved borders. Look at lines that are straight in the world (board edges, door frames): they should be
straighter on the right. The chosen calibration is saved to `artifacts/calibration.npz` for Task 3.

In [ ]:
display(Image(out_1a / 'before_after.jpg', width=900))
saved = np.load(cfg_1a.paths.artifacts / 'calibration.npz')
print(cfg_1a.paths.artifacts / 'calibration.npz', 'keys:', saved.files)

### 1A summary
- **Evidence for rubric 1.1:** K and the 5 distortion coefficients (table above), reprojection error overall and per
  view, corner overlays, the before/after image, and the saved `calibration.npz`.
- **Improvement shown:** baseline vs `fix_k3` with the same data, so the report can argue for the less overfitted model
  with numbers.
- **Limitations:** the result is valid only for this camera at this resolution and zoom; refocusing or changing the
  resolution needs a new calibration (or a scaled K).

## 1B Classical processing — [1B.1] … [1B.5]

### Goal
Show the core classical tools on real images and measure them: denoising filters (rubric 1.2), the frequency domain
(1.2), gradients and edges (1.2), and interest points and matching between views (1.3).

### Inputs and outputs
| | |
| --- | --- |
| Input | `data/samples/clean.jpg` (one sharp photo; noise is added in code so the clean truth is known), `pair_1.jpg` + `pair_2.jpg` (same scene, two views), `unrelated.jpg` (different scene). Sources: `data/samples/SOURCES.md` |
| Output | `results/task1b/`: figures `01_…05_…`, `metrics_filtering.csv`, `metrics_matching.csv`, `metrics.json` |

### Pipeline
`[1B.1] add noise` → `[1B.2] box / Gaussian / median` → `[1B.2m] MSE, PSNR` → `[1B.3] FFT filtering` →
`[1B.4] Sobel + Canny` → `[1B.5h] Harris` → `[1B.5] ORB / SIFT / Harris matching + ratio test + RANSAC`.
All settings (kernel size, sigma, cut-off radius, thresholds) come from `cfg.task1b`.

In [ ]:
s = cfg.task1b
inputs = {name: read_image(cfg.project_root / getattr(s, name), s.max_side_px)
          for name in ('sample_image', 'pair_image_1', 'pair_image_2', 'unrelated_image')}
fig = show_grid(list(inputs.values()), [f'{k}  {v.shape[1]}x{v.shape[0]}' for k, v in inputs.items()], cols=4,
                figsize=(16, 3.6))
plt.show()
print(open(cfg.project_root / 'data/samples/SOURCES.md', encoding='utf-8').read().split('Why these')[0])

### Run [1B.R]

In [ ]:
from src.task1b import run_task1b

report_1b = run_task1b(cfg)
out_1b = cfg.paths.results / 'task1b'
print('Wrote', len(report_1b.figure_paths), 'figures and metrics to', out_1b)

### [1B.1]–[1B.2m] Noise and spatial filtering

**Noise models.** *Gaussian* noise adds $n \sim \mathcal N(0, \sigma^2)$ to every pixel ($\sigma = 20$ on the 0–255
scale): it models sensor/thermal noise. *Salt-and-pepper* sets a fraction of pixels (5 %) to 0 or 255: it models dead
pixels and transmission errors. A fixed seed makes the experiment repeatable.

**Filters** (same $k \times k$ window, $k = 5$):
- **Box** – plain average of the window. Linear; blurs edges as much as noise.
- **Gaussian** – weighted average with weights $\propto e^{-(x^2+y^2)/2\sigma^2}$, so near pixels count more. Linear;
  the best linear choice for Gaussian noise but still blurs edges.
- **Median** – the middle value of the window. Non-linear: one extreme pixel cannot move the median, so impulse noise
  disappears while edges stay sharp.

**Scores** against the clean image: $\mathrm{MSE} = \frac{1}{N}\sum (I - \hat I)^2$ and
$\mathrm{PSNR} = 10 \log_{10}(255^2 / \mathrm{MSE})$ dB. Higher PSNR = closer to the clean image; +3 dB ≈ half the error.
The `noisy` row is the starting point each filter must beat.

In [ ]:
scores = pd.read_csv(out_1b / 'metrics_filtering.csv')
display(scores.pivot(index='method', columns='noise', values='psnr_db').round(2))
display(Image(out_1b / '01_filter_comparison.png'), Image(out_1b / '01b_filter_psnr.png'))

In [ ]:
# Conclusion (computed)
lines = []
for noise, group in scores.groupby('noise', sort=False):
    start = group[group.method == 'noisy'].psnr_db.item()
    ranked = group[group.method != 'noisy'].sort_values('psnr_db', ascending=False)
    best, worst = ranked.iloc[0], ranked.iloc[-1]
    lines.append(f"- **{noise}:** best = **{best.method}** at {best.psnr_db:.2f} dB (+{best.psnr_db - start:.2f} dB over "
                 f"the noisy image); weakest = {worst.method} at {worst.psnr_db:.2f} dB.")
check = __import__('json').loads((out_1b / 'metrics.json').read_text())['filter_scratch_vs_opencv_max_abs_diff']
lines.append(f"- Own convolution (`method='scratch'`) vs OpenCV: max difference {max(check.values())} grey level(s) "
             "– the from-scratch implementation is correct (differences are rounding only).")
display(Markdown('**Conclusion.**\n' + '\n'.join(lines) + '\n\n'
    'Why: Gaussian noise is spread over every pixel with zero mean, so a weighted average cancels it best and the '
    'Gaussian filter wins. Salt-and-pepper noise is a few extreme outliers; an average spreads each outlier into its '
    'neighbours, while the median ignores it, so the median wins clearly. The price of every filter is blur: '
    'fine texture is lost even where there was no noise, which is why no filter reaches the clean image.'))

### [1B.3] Frequency domain (FFT)

The 2D Fourier transform writes the image as a sum of sinusoids: $F(u,v) = \sum_{x,y} I(x,y)\,e^{-2\pi i(ux/W + vy/H)}$.
- After `fftshift` the **DC term** (average brightness, $u=v=0$) is in the **centre**; distance from the centre =
  spatial frequency. Smooth areas live near the centre, edges and fine texture far out.
- We show $\log(1 + |F|)$ because the magnitudes span many orders of magnitude.
- Filtering = multiply $F$ by a mask, then `ifftshift` + inverse FFT. A **low-pass** mask keeps the centre (blur), a
  **high-pass** mask removes it (edges/detail only; the result is signed and averages to 0, so it is shown on a
  symmetric blue–red scale).
- **Ideal** mask: hard circle of radius $r$. Its inverse transform is a sinc-like ripple, so the result shows
  **ringing** (ghost copies of edges). **Gaussian** $e^{-D^2/2r^2}$ and **Butterworth** $1/(1+(D/r)^{2n})$ masks fall
  off smoothly and avoid it.

Multiplying in the frequency domain is the same as convolving in the image domain (convolution theorem), so the
Gaussian low-pass here is the same idea as the Gaussian blur in [1B.2], just applied globally.

In [ ]:
display(Image(out_1b / '02_frequency_domain.png'), Image(out_1b / '02b_fft_radius_sweep.png'))

**How to read it.** In the spectrum, the bright cross through the centre comes from the image borders (the FFT treats
the image as periodic, so the left/right and top/bottom edges act as strong vertical/horizontal edges); bright streaks
point perpendicular to strong straight edges in the scene. In the radius sweep, a smaller $r$ keeps fewer frequencies:
more blur, and for the ideal mask wider ringing.

**Conclusion.** The low-pass results confirm that most of the visible structure is carried by a small central part of
the spectrum (the cut-off radius `fft_radius` is a few tens of pixels out of several hundred), while the high-pass result keeps only edges and texture. At the same
cut-off, the ideal mask produces visible ringing around high-contrast edges while the Gaussian and Butterworth masks
give a clean blur, which is why smooth masks are preferred in practice.

### [1B.4] Gradients and edges

**Sobel.** Derivative kernels on a smoothed image:
$G_x = \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix} * I$, $G_y = G_x^T * I$.
$G_x$ measures change **across columns**, so it responds to **vertical** edges; $G_y$ to **horizontal** edges. Both are
signed (dark→bright vs bright→dark), shown in blue/red. Magnitude $\sqrt{G_x^2+G_y^2}$ is edge strength, orientation
$\operatorname{atan2}(G_y, G_x)$ is edge direction.

**Canny** turns the gradient into thin, connected edges: (1) Gaussian smoothing, (2) Sobel gradient, (3) non-maximum
suppression – keep only the ridge of the gradient across the edge, (4) hysteresis – pixels above `high` start an edge,
pixels above `low` continue it only when connected to a strong one. The advanced variant sets `low, high` to
0.67× and 1.33× the median grey level, so it adapts to the image's brightness.

In [ ]:
display(pd.DataFrame(report_1b.edge_stats))
display(Image(out_1b / '03_edges.png'), Image(out_1b / '03b_canny_sweep.png'))

In [ ]:
# Conclusion (computed)
edges = pd.DataFrame(report_1b.edge_stats).set_index('variant')
lo, mid, hi = edges.edge_density.iloc[0], edges.edge_density.iloc[1], edges.edge_density.iloc[2]
auto = edges.loc['auto (median)']
display(Markdown(
    f"**Conclusion.** Raising the Canny thresholds from {tuple(cfg.task1b.canny_sweep[0])} to "
    f"{tuple(cfg.task1b.canny_sweep[2])} cuts the edge pixels from {lo * 100:.2f} % to {hi * 100:.2f} % of the image "
    f"({lo / max(hi, 1e-9):.1f}× fewer). Low thresholds keep texture and noise as false edges; high thresholds keep only "
    f"strong object boundaries but break weak ones. The default (80, 160) gives {mid * 100:.2f} %, and the automatic "
    f"thresholds ({int(auto.low)}, {int(auto.high)}) give {auto.edge_density * 100:.2f} % without hand tuning. "
    "Gx and Gy confirm the theory: vertical structures appear in Gx, horizontal ones in Gy."))

### [1B.5h] Harris corners

Around each pixel, the **structure tensor** sums the gradient products in a small window:
$M = \sum_w \begin{bmatrix} I_x^2 & I_xI_y \\ I_xI_y & I_y^2 \end{bmatrix}$. Its eigenvalues say how much the
brightness changes when the window shifts: both small = flat, one large = edge, **both large = corner**. Harris avoids
computing eigenvalues with $R = \det M - k\,(\operatorname{trace} M)^2$, $k = 0.04$: $R \gg 0$ corner, $R < 0$ edge,
$|R|$ small flat. We keep local maxima of $R$ above 3.5 % of the maximum, strongest 80. The advanced version computes
$R$ from the formula (`method='scratch'`) and is compared with `cv2.cornerHarris`.

In [ ]:
print('Harris corners kept:', report_1b.harris_stats)
display(Image(out_1b / '04_harris.png'))

In [ ]:
# Conclusion (computed)
h = report_1b.harris_stats
display(Markdown(
    f"**Conclusion.** OpenCV and the from-scratch Harris keep {h['opencv']} and {h['scratch']} corners and mark the "
    "same kind of places: corners of buildings, objects and texture, where brightness changes in two directions. "
    "Long straight edges and flat regions get no corners, as the theory predicts. Harris gives locations "
    "only; to match them between images they still need a descriptor, which is what the next step adds."))

### [1B.5] Interest points and matching between two views

Three detector/descriptor combinations, same matching rules:
| Detector | Keypoints | Descriptor | Distance |
| --- | --- | --- | --- |
| **ORB** (baseline) | FAST corners at several scales, with orientation | rotated BRIEF, 256 binary tests | Hamming |
| **SIFT** | extrema of Difference-of-Gaussians across scales | 128-d histogram of gradient orientations | Euclidean (L2) |
| **Harris + ORB** | Harris corners from [1B.5h] (single scale) | ORB descriptor | Hamming |

**Lowe's ratio test.** For each descriptor in image 1 find its two nearest neighbours in image 2 and keep the match
only if $d_1 < 0.75\,d_2$. It rejects **ambiguous** matches – repeated patterns (the identical arches here) where the
best candidate is barely better than the second.

**RANSAC homography (advanced).** Repeatedly pick 4 matches, fit a homography $H$, count matches that land within
5 px; keep the $H$ with most **inliers**. Valid here because the pair was taken while the camera rotated about one
point (a panorama pair), and two such views are related exactly by a homography. Inlier ratio = how many ratio-test
matches are geometrically consistent.

**Negative control.** The same matching against an unrelated scene shows how many matches appear by chance.

In [ ]:
matching = pd.read_csv(out_1b / 'metrics_matching.csv')
display(matching)
for path in report_1b.figure_paths:
    if path.name.startswith('05_matches_same_scene'):
        print(path.name, '  (green = RANSAC inlier, red = outlier)')
        display(Image(path, width=950))
display(Image(out_1b / '05_matches_unrelated_sift.png', width=950))

In [ ]:
# Conclusion (computed)
same = matching[matching.pair == 'same_scene'].set_index('detector')
other = matching[matching.pair == 'unrelated'].set_index('detector')
best = same.ransac_inliers.idxmax()
fastest = same.time_ms.idxmin()
lines = [f"- Same scene: **{best}** gives the most geometrically verified matches ({same.loc[best, 'ransac_inliers']} "
         f"inliers, {same.loc[best, 'inlier_ratio'] * 100:.0f} % of its ratio-test matches)."]
for det in same.index:
    lines.append(f"  - {det}: {same.loc[det, 'good_matches']} ratio-test matches → {same.loc[det, 'ransac_inliers']} inliers "
                 f"({same.loc[det, 'inlier_ratio'] * 100:.0f} %), {same.loc[det, 'time_ms']:.0f} ms")
lines.append(f"- Speed: **{fastest}** is fastest ({same.loc[fastest, 'time_ms']:.0f} ms); SIFT is "
             f"{same.loc['sift', 'time_ms'] / same.loc['orb', 'time_ms']:.1f}× slower than ORB because of its float "
             "descriptors and scale-space search. Harris + ORB is fast too, but it detects at one scale only, so it "
             "breaks down when the zoom or distance changes between views.")
if len(other):
    lines.append(f"- Unrelated scene: only {int(other.good_matches.min())}–{int(other.good_matches.max())} ratio-test matches "
                 f"(vs {int(same.good_matches.min())}–{int(same.good_matches.max())} on the same scene), so the ratio test "
                 "already separates related from unrelated images. Their inlier ratios are not meaningful: with "
                 "fewer than about 10 matches, 4 random points always fit some homography – use the inlier **count** "
                 "as the decision, e.g. 'same scene if ≥ 20 inliers'.")
display(Markdown('**Conclusion.**\n' + '\n'.join(lines) + '\n\n'
    'Overall: the ratio test removes ambiguous matches, and RANSAC removes the remaining geometrically wrong ones '
    '(red lines in the figures). ORB is the practical choice for the real-time app; SIFT is the more robust choice '
    'when the viewpoint or scale changes a lot.'))

### Questions from the workshop pack, answered
1. **Why can Gaussian filtering give a higher MSE than the noisy image?** Smoothing also changes clean pixels: on an image
   with many sharp edges and little noise, the blur error at the edges can exceed the noise it removes.
2. **Where is the DC term after `fftshift`?** At the centre of the spectrum, pixel $(H/2, W/2)$.
3. **Why does Sobel $G_x$ highlight vertical edges?** It differentiates along x (across columns); a vertical edge is a
   change in brightness from left to right.
4. **Which matches does Lowe's ratio test remove?** Ambiguous ones, whose best and second-best candidates are almost
   equally close – typically repeated textures – so the "best" is likely wrong.

### Task 1 summary
| Rubric | Evidence in this notebook |
| --- | --- |
| 1.1 Calibration | K, distortion, reprojection error per view, baseline vs fix_k3, before/after, `calibration.npz` |
| 1.2 Filtering, FFT, edges | PSNR table for 3 filters × 2 noise types, own convolution check, FFT masks and radius sweep, Sobel/Canny with threshold sweep and auto thresholds |
| 1.3 Features | Harris (OpenCV and from scratch), ORB vs SIFT vs Harris, ratio test, RANSAC inliers, unrelated-scene control |

## 2 Deep learning model and data pipeline — [2.0] … [2.8]

### Goal
Requirement (c): the assistant must *understand* images with a trained deep-learning model. Task 2 builds the full
path from raw downloaded photos to a saved model that Task 3 can call: a documented **data pipeline** (rubric 2.1), a
justified **model** (2.2) and an honest **evaluation** (2.3). It also compares several models and tunes them, so the
choice of the deployed model is backed by numbers: quality, size, speed and training time.

### Dataset and justification
**TensorFlow Flowers (`tf_flowers`)** – 3,670 real Flickr photos of 5 flower species (daisy, dandelion, roses,
sunflowers, tulips), licensed CC-BY 2.0 (every photographer is credited in `data/raw/flower_photos/LICENSE.txt`).
Source: `storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz` (TensorFlow team).

- **Fits the scenario**: an *organic farm assistant* that recognises which plant/flower is in front of the camera
  (crop scouting, pollinator planting, weed vs crop).
- **Realistic and messy**: different cameras, sizes (143–1024 px), lighting, backgrounds, several flowers or people in
  a frame, a few near-duplicate photos – exactly what a data pipeline has to clean.
- **Right size for the coursework**: roughly balanced classes, small enough to train on a laptop CPU, large enough
  for a real train/validation/test split. The pipeline is generic (one folder per class), so another dataset such as
  Kaggle's *fruits fresh vs rotten* can be dropped into `data/raw/` without code changes.

### Pipeline (one function per step, all settings in `cfg.task2`)
`[2.0] download` → `[2.1] ingest (manifest)` → `[2.2] clean` → `[2.3] stratified split` → `[2.4] transforms,
augmentation, loaders` → `[2.5] model` → `[2.6t] tune` → `[2.6] train` → `[2.7] evaluate + [2.7b] benchmark` →
`[2.8] save` → `[3.0]/[3.3] load and predict`.

| Output | Where |
| --- | --- |
| Model for Task 3 | `artifacts/model.pth` + `artifacts/classes.json` |
| Split lists | `data/splits/train.csv`, `val.csv`, `test.csv` |
| Evidence | `results/task2/`: `manifest_raw.csv`, `removal_log.csv`, `tuning_*.csv`, `experiments.csv`, figures, `metrics.json` |

In [ ]:
# Training all experiments takes about 55 minutes on a 4-core CPU, about 10-15 minutes on a Colab T4 GPU.
# RETRAIN = IN_COLAB          # on Colab: retrain on the GPU; locally: reuse the saved results reuses the saved results/model; set True (or delete results/task2/metrics.json) to train again.
import json
from src.train import run_task2

RETRAIN = IN_COLAB          # on Colab: retrain on the GPU; locally: reuse the saved results
out_2 = cfg.paths.results / 'task2'
if RETRAIN or not (out_2 / 'metrics.json').is_file():
    run_task2(cfg)
m2 = json.loads((out_2 / 'metrics.json').read_text(encoding='utf-8'))
resolve = lambda p: Path(p) if Path(p).is_absolute() else cfg.project_root / p
print('Classes:', m2['classes'])
print('Selected model for Task 3:', m2['selected'], '->', m2['model_path'])

### [2.1] Ingestion
`build_manifest` walks `data/raw/<class>/` and records, for every file: path, label (= folder name), width, height,
colour mode, file size and whether it **fully decodes** (`PIL.Image.load()`), so a broken file is found here and not
in the middle of an epoch. The manifest is the record of the raw data before anything is changed.

In [ ]:
raw = pd.read_csv(out_2 / 'manifest_raw.csv')
summary = raw.groupby('label').agg(images=('path', 'size'), median_w=('width', 'median'), median_h=('height', 'median'),
                                   min_side=('width', lambda s: int(min(s.min(), raw.loc[s.index, 'height'].min()))),
                                   unreadable=('readable', lambda s: int((~s).sum())))
display(summary)
display(Markdown(
    f"**Conclusion.** {len(raw)} images in {raw.label.nunique()} classes; sizes range from "
    f"{raw[['width', 'height']].min().min()} to {raw[['width', 'height']].max().max()} px (median "
    f"{int(raw.width.median())}×{int(raw.height.median())}), colour modes {({k: int(v) for k, v in raw['mode'].value_counts().items()})}, "
    f"{int((~raw.readable).sum())} unreadable files. The largest class has "
    f"{summary.images.max() / summary.images.min():.2f}× the images of the smallest, a mild imbalance: accuracy "
    "is still meaningful, but macro F1 is reported too so the smaller classes count equally."))

### [2.2] Cleaning
Rules, applied in order, each removal written to `removal_log.csv` with its reason:
1. **Unreadable / corrupt** files.
2. **Too small** (`min_side` = 64 px): too little detail, upscaling would only add blur.
3. **Odd shape** (aspect ratio > 3): panoramas/strips that a square crop would mostly cut away.
4. **Duplicates** with a perceptual **difference hash (dHash)**: shrink to 9×8 grey pixels, 64 bits =
   "is this pixel brighter than its right neighbour". Re-saved, resized or re-compressed copies keep almost the same
   bits, so a Hamming distance ≤ 4 bits means "same photo". Same label → keep the largest copy. **Different labels →
   drop all copies** (the label is unreliable).

Why duplicates matter: if one copy lands in *train* and another in *test*, the model is tested on a photo it has
memorised and the score is inflated (**data leakage**). Cleaning runs before the split for that reason.
Greyscale/CMYK/palette images are not removed; they are converted to RGB when loaded ([2.4c]).

In [ ]:
removed = pd.read_csv(out_2 / 'removal_log.csv')
removed = removed[removed.reason != 'nothing removed']
reason_kind = removed.reason.str.replace(r' of .*| \[.*', '', regex=True)
display(reason_kind.value_counts().rename('images removed').to_frame())
pairs = removed[removed.reason.str.startswith('duplicate of')].head(4)
images, titles = [], []
for _, row in pairs.iterrows():
    kept = next(Path(resolve(row.path)).parent.glob(row.reason.replace('duplicate of ', '')))
    images += [read_image(resolve(row.path)), read_image(kept)]
    titles += [f'removed: {Path(row.path).name}', f'kept: {kept.name}']
if images:
    fig = show_grid(images, titles, cols=4, figsize=(14, 3.2 * ((len(images) + 3) // 4)),
                    suptitle='[2.2] Near-duplicates found by dHash (removed | kept)')
    plt.show()

In [ ]:
display(Markdown(
    f"**Conclusion.** {len(removed)} of {len(raw)} images ({len(removed) / len(raw) * 100:.1f} %) were removed: "
    + ', '.join(f'{n} {k}' for k, n in reason_kind.value_counts().items())
    + f". {m2['clean_images']} clean images remain. The dataset is already curated, so few files go, but the "
    "duplicate check still finds the same photo saved twice, and photos filed under two different classes; without "
    "this step those copies could be split across train and test and inflate the score."))

### [2.3] Stratified split
70 % train / 15 % validation / 15 % test, **stratified** by class (`sklearn.train_test_split(stratify=label)`), so
every split has the same class proportions, with a fixed seed. Roles:
- **train**: the only data the weights learn from (and the only split that is augmented);
- **validation**: picks the best epoch, the hyperparameters and the model to deploy;
- **test**: used once at the end for the reported score. Choosing anything on test would make the test score optimistic.

The split is done **before** augmentation and **after** de-duplication, and saved to `data/splits/*.csv` so it can be
reproduced and inspected.

In [ ]:
counts = pd.DataFrame(m2['class_counts'])
counts['total'] = counts.sum(axis=1)
shares = (counts[['train', 'val', 'test']].div(counts[['train', 'val', 'test']].sum()) * 100).round(1)
display(counts, shares.add_suffix(' %'))
display(Image(out_2 / '01_class_counts.png'))
spread = (shares.max(axis=1) - shares.min(axis=1)).max()
display(Markdown(
    f"**Conclusion.** {counts.train.sum()} / {counts.val.sum()} / {counts.test.sum()} images in train / val / test. "
    f"Each class has the same share of every split to within {spread:.1f} percentage points, so validation and test "
    "are representative of the training distribution and per-class scores are comparable."))

### [2.4] Transforms, augmentation and loaders
**Preprocessing (all splits, and inference in Task 3):** resize the short side to 1.14 × size, centre crop to
size × size, convert to a tensor in [0, 1], normalise each channel with the ImageNet mean/std (required by the
pretrained model; harmless for the CNN). One function, `make_transforms(train=False)`, is shared by evaluation and
`inference.predict`, so the app sees images exactly as the model did during validation.

**Augmentation (train only)** – new random variants every epoch, so the model cannot memorise exact pixels:
| Transform | Simulates |
| --- | --- |
| RandomResizedCrop (70–100 % of the area) | different distances / framing (scaling) |
| Horizontal flip | flowers have no left/right |
| Rotation ±15° | tilted camera |
| Colour jitter (brightness, contrast, saturation ±25 %, hue ±3 %) | lighting and camera colour differences |
| Gaussian noise (σ = 0.03, p = 0.5) | sensor noise, low light |

Hue is kept small on purpose: colour is a real cue between classes (yellow dandelion vs red rose).

In [ ]:
display(Image(out_2 / '02_class_examples.png', width=650))
display(Image(out_2 / '03_augmentation_examples.png', width=900))

**Conclusion.** The examples show how varied the raw photos are (close-ups, fields, bouquets, people). Each augmented
copy is still clearly the same flower, so the label stays correct while position, scale, angle, colour and noise
change – the variations the app will meet with real users' photos. The effect of augmentation is measured below by
training the same CNN with and without it.

### [2.5] Models
**Baseline – `SmallCNN` (from scratch).** 4 blocks of Conv 3×3 → BatchNorm → ReLU → MaxPool 2×2 with
32 → 64 → 128 → 256 channels, then global average pooling, dropout 0.3 and one linear layer.
Each block halves the resolution and doubles the channels: early filters learn edges and colour blobs, later ones
petal and flower-head patterns. Global average pooling instead of a large flatten layer keeps it at ~0.4 M parameters.

**Advanced – `MobileNetV3-Large` with transfer learning.** A network designed for phones (Howard et al., 2019):
depthwise-separable convolutions (a per-channel 3×3 conv followed by a 1×1 conv, ~8–9× cheaper than a normal conv),
inverted residual blocks, squeeze-and-excitation channel attention and the h-swish activation. Its weights come from
training on **ImageNet** (1.28 M images, 1000 classes), so its filters already detect edges, textures, petals and
object parts. We replace the 1000-class head with a new 5-class head and train in two stages:
1. **Feature extraction** (`freeze_epochs`): backbone frozen, only the new head learns, so the random head cannot send
   large gradients into the good pretrained filters.
2. **Fine-tuning**: everything is unfrozen with a smaller learning rate (`fine_tune_lr`) to adapt the filters gently.

**Training setup (both):** cross-entropy with label smoothing 0.1 (targets 0.92/0.02 instead of 1/0, which reduces
over-confidence), AdamW (Adam + decoupled weight decay 1e-4), cosine learning-rate decay, and **best-epoch selection
on validation macro F1** (a form of early stopping).

In [ ]:
from src.model import build_model, count_parameters
for name in ('small_cnn', 'mobilenet_v3'):
    print(f'{name:13s} {count_parameters(build_model(name, len(m2["classes"]), pretrained=False)) / 1e6:.2f} M parameters')
print(build_model('small_cnn', len(m2['classes'])))

### [2.6t] Hyperparameter tuning
The learning rate is the setting with the largest effect, so it is tuned for both trainable designs with a small
**grid search**: every value is trained for a short budget (`tune_epochs` = 5) with the same seed and data, and
compared on **validation** macro F1 only. The best value is then used for the full training run.
- `cnn_aug`: learning rate ∈ {0.003, 0.001, 0.0003}
- `mobilenet_transfer`: fine-tuning learning rate ∈ {0.001, 0.0003, 0.0001}

Short runs rank settings reliably at a fraction of the cost (3 short runs ≈ one full run). The curves show *how* each
value learns: too high is noisy or unstable, too low learns slowly.

In [ ]:
for name, tune in m2['tuning'].items():
    display(Markdown(f'**{name}** – grid {tune["grid"]}'))
    display(pd.read_csv(out_2 / f'tuning_{name}.csv'))
    display(Image(out_2 / f'04a_tuning_{name}.png'))

In [ ]:
lines = []
for name, tune in m2['tuning'].items():
    trials = sorted(tune['trials'], key=lambda t: -t['val_macro_f1'])
    key = list(tune['grid'])[0]
    lines.append(f"- **{name}**: best {key} = {tune['chosen'][key]} (val macro F1 {trials[0]['val_macro_f1']:.3f}); "
                 f"worst {key} = {trials[-1][key]} ({trials[-1]['val_macro_f1']:.3f}), a spread of "
                 f"{trials[0]['val_macro_f1'] - trials[-1]['val_macro_f1']:.3f}. Each trial took "
                 f"{np.mean([t['seconds'] for t in trials]) / 60:.1f} min.")
spreads = [max(t['val_macro_f1'] for t in v['trials']) - min(t['val_macro_f1'] for t in v['trials']) for v in m2['tuning'].values()]
display(Markdown('**Conclusion (computed).**\n' + '\n'.join(lines) + f"\n\nWithin the same short budget a poor learning rate "
                 f"costs up to {max(spreads) * 100:.1f} F1 points, so tuning is worth its cost. The pretrained model is more "
                 "sensitive to the fine-tuning rate than the CNN is to its rate, and every value already beats the CNN, "
                 "which shows the pretrained features do most of the work. Tuning used validation only, so the test split is still unseen."))

### [2.6] Training curves
For each experiment: loss (left) and accuracy / macro F1 (right) per epoch on train and validation. The dotted line
marks the epoch whose weights were kept. How to read them:
- both losses falling together → still learning; **validation loss rising while train loss falls → overfitting**;
- a large gap between train and validation accuracy → memorising the training set.

In [ ]:
exps = pd.DataFrame(m2['experiments']).set_index('experiment')
for name in exps.index:
    display(Image(out_2 / f'04_history_{name}.png'))
lines = []
for name, h in m2['histories'].items():
    best = h['best_epoch'] - 1
    gap = h['train_acc'][-1] - h['val_acc'][-1]
    lines.append(f"- **{name}**: best epoch {h['best_epoch']}/{len(h['val_acc'])}, val accuracy "
                 f"{h['val_acc'][best]:.3f}; final train–val accuracy gap {gap:+.3f}"
                 + (' → clear overfitting' if gap > 0.15 else ' → mild overfitting' if gap > 0.05 else ' → no overfitting') + '.')
display(Markdown('**Conclusion (computed).**\n' + '\n'.join(lines)))

### [2.7] Evaluation on the test split
Metrics (test split, touched once):
- **Accuracy** = correct / all.
- **Precision** (per class) = of the images predicted as class *c*, how many are *c*; **recall** = of the images of
  class *c*, how many were found; **F1** = harmonic mean of the two.
- **Macro F1** = average F1 over classes, every class weighted equally (robust to the mild imbalance).
- **Confusion matrix**: rows = true class, columns = predicted; off-diagonal cells show which classes are confused.

In [ ]:
quality = exps[['model', 'image_size', 'augment', 'val_accuracy', 'val_macro_f1', 'test_accuracy', 'test_macro_f1']]
display(quality)
for name in exps.index:
    display(Image(out_2 / f'05_confusion_{name}.png', width=420))
sel = m2['selected']
per_class = pd.DataFrame(m2['test_metrics'][sel]['per_class']).T.round(3)
display(Markdown(f'**Per-class scores of the selected model `{sel}`**'), per_class)
display(Image(out_2 / f'06_misclassified_{sel}.png'))

In [ ]:
confusion = np.array(m2['test_metrics'][sel]['confusion'])
off = confusion.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
weakest = per_class.f1.idxmin()
aug_gain = exps.loc['cnn_aug', 'test_macro_f1'] - exps.loc['cnn_no_aug', 'test_macro_f1'] if {'cnn_aug', 'cnn_no_aug'} <= set(exps.index) else None
tl_gain = exps.loc['mobilenet_transfer', 'test_macro_f1'] - exps.loc['cnn_aug', 'test_macro_f1'] if {'mobilenet_transfer', 'cnn_aug'} <= set(exps.index) else None
lines = [f"- Selected `{sel}` (best **validation** macro F1): test accuracy {exps.loc[sel, 'test_accuracy']:.3f}, "
         f"test macro F1 {exps.loc[sel, 'test_macro_f1']:.3f}."]
if aug_gain is not None:
    h_aug, h_plain = m2['histories']['cnn_aug'], m2['histories']['cnn_no_aug']
    if aug_gain > 0:
        lines.append(f"- **Augmentation** improves the from-scratch CNN's test macro F1 by {aug_gain:+.3f}: it reduces "
                     "overfitting and generalises better.")
    else:
        lines.append(f"- **Augmentation** changes the from-scratch CNN's test macro F1 by {aug_gain:+.3f}. This is a training-"
                     f"budget effect, not a sign that augmentation hurts: the augmented model was still improving (best epoch "
                     f"{h_aug['best_epoch']}/{len(h_aug['val_acc'])}) and its train accuracy ({h_aug['train_acc'][-1]:.3f}) is "
                     f"below its validation accuracy, i.e. it is *under*-fitting because every epoch shows it harder, "
                     f"new variants. The plain CNN reaches its best at epoch {h_plain['best_epoch']} because it sees the same "
                     "pixels each time. With more epochs the augmented model is expected to overtake; with only ~2.5 k images "
                     "the larger gain comes from transfer learning.")
if tl_gain is not None:
    lines.append(f"- **Transfer learning** adds {tl_gain:+.3f} macro F1 over the best from-scratch CNN: ImageNet features "
                 "transfer to flowers, so ~2.5 k training images are enough.")
lines.append(f"- Most frequent confusion: **{m2['classes'][i]} → {m2['classes'][j]}** ({off[i, j]} test images); "
             f"weakest class: **{weakest}** (F1 {per_class.loc[weakest, 'f1']:.3f}).")
lines.append("- The most confident mistakes above are mostly hard cases rather than model bugs: several flower types in "
             "one photo, the flower small or out of focus, or colours shared between classes (red tulips vs roses).")
display(Markdown('**Conclusion (computed).**\n' + '\n'.join(lines)))

### [2.7b] Which model suits best? Quality vs size vs speed vs training time
The app runs on a normal laptop CPU, so accuracy alone does not decide. Each model is also measured for:
- **Parameters** and **size on disk** (MB) – memory and download size;
- **Latency** – median CPU time to classify one image (what a user waits for in the app);
- **Throughput** – images per second in batches of 32 (bulk processing, e.g. a whole warehouse scan);
- **Training time** – total minutes and seconds per epoch on this machine (4-core CPU).

In [ ]:
cost = exps[['test_macro_f1', 'test_accuracy', 'parameters_M', 'size_MB', 'latency_ms', 'throughput_img_s',
             'train_minutes', 'sec_per_epoch', 'best_epoch', 'epochs']]
display(cost)
display(Image(out_2 / '07_model_tradeoff.png', width=700))

In [ ]:
best_q = cost.test_macro_f1.idxmax(); fastest = cost.latency_ms.idxmin(); smallest = cost.size_MB.idxmin()
same_net = exps.groupby('model').latency_ms.agg(['min', 'max'])
cheapest = cost.train_minutes.idxmin()
REALTIME_MS = 100   # a reply within 0.1 s feels instant in a UI
ok = cost[cost.latency_ms <= REALTIME_MS]
pick = ok.test_macro_f1.idxmax() if len(ok) else best_q
lines = [f"- Best quality: **{best_q}** (macro F1 {cost.loc[best_q, 'test_macro_f1']:.3f}).",
         f"- Fastest: **{fastest}** ({cost.loc[fastest, 'latency_ms']:.1f} ms/image, "
         f"{cost.loc[fastest, 'throughput_img_s']:.0f} img/s batched; the two CNN runs are the same network, so their "
         f"{same_net.loc['small_cnn', 'min']:.1f}–{same_net.loc['small_cnn', 'max']:.1f} ms difference is timing noise); smallest: **{smallest}** "
         f"({cost.loc[smallest, 'size_MB']:.1f} MB); cheapest to train: **{cheapest}** ({cost.loc[cheapest, 'train_minutes']:.1f} min).",
         f"- Quality gap best vs smallest: {cost.loc[best_q, 'test_macro_f1'] - cost.loc[smallest, 'test_macro_f1']:+.3f} "
         f"macro F1 for {cost.loc[best_q, 'size_MB'] / cost.loc[smallest, 'size_MB']:.0f}× the size and "
         f"{cost.loc[best_q, 'latency_ms'] / cost.loc[smallest, 'latency_ms']:.1f}× the latency.",
         f"- **Recommendation for the Task 3 app:** `{pick}` – the most accurate model that still answers within "
         f"{REALTIME_MS} ms on a CPU ({cost.loc[pick, 'latency_ms']:.1f} ms). "
         + ("It is also the model selected on validation and saved to `artifacts/model.pth`." if pick == m2['selected'] else
            f"Note: `{m2['selected']}` was saved because it has the best validation score; switch if latency matters more."),
         f"- For a very small device (microcontroller, old phone) the from-scratch CNN is the fallback: "
         f"{cost.loc[smallest, 'size_MB']:.1f} MB, {cost.loc[smallest, 'latency_ms']:.1f} ms, at a cost in accuracy."]
display(Markdown('**Conclusion (computed).**\n' + '\n'.join(lines)))

### [2.8] Saving and calling the model (the bridge to Task 3)
`save_model` writes the weights plus everything needed to rebuild the model: class names, model name and input size
(`artifacts/classes.json`). `inference.load_model` rebuilds the network without downloading ImageNet weights and
`inference.predict` applies the same evaluation transform. Below, the saved artifact is loaded exactly as the app will
load it and used on random test images.

In [ ]:
from src.inference import load_model, predict
model_2, meta_2 = load_model(cfg.paths.artifacts / 'model.pth', cfg.paths.artifacts / 'classes.json')
test_split = pd.read_csv(cfg.paths.splits / 'test.csv').sample(8, random_state=3)
images, titles = [], []
for path, label in zip(test_split.path, test_split.label):
    bgr = read_image(resolve(path))
    pred = predict(model_2, meta_2, bgr)
    images.append(bgr)
    titles.append(f"true {label} | pred {pred.label} ({pred.score:.2f})" + ('' if pred.label == label else '  ✗'))
fig = show_grid(images, titles, cols=4, figsize=(14, 6.4), suptitle='[3.3] Predictions from artifacts/model.pth')
plt.show()
print('model:', meta_2['model_name'], '| input', meta_2['image_size'], 'px | classes', meta_2['classes'])

### [2.9] Baselines: is deep learning worth it?
A deep model's score means little without reference points. Every method below uses the **same split** (train to fit,
validation to choose settings, test scored once) and the same metrics, so each row is directly comparable.

| Family | Method | Idea |
| --- | --- | --- |
| Chance | **Majority class** (the classification "mean" model) | always predicts the most common class – the floor any real model must beat |
| Chance | **Random by class share** | guesses in proportion to class frequencies |
| Classical CV + ML | **k-NN, logistic regression, SVM (RBF), random forest** on a hand-crafted descriptor | the pre-deep-learning approach: describe each image with fixed features, then learn a classifier |
| Pretrained features + ML | **logistic regression / SVM on frozen ImageNet MobileNetV3 features** ("linear probe") | deep features without any training on flowers: isolates what fine-tuning adds |
| Deep learning | the CNNs and fine-tuned MobileNetV3 from [2.6] | features learned end to end |

**Hand-crafted descriptor** (2,276 numbers per image, built from Task 1 techniques, centre crop 128 px):
- **colour** – HSV histogram 16 hue × 4 saturation × 4 value bins (256): *which colours* are present;
- **shape** – HOG, written from scratch: Sobel gradients → 9-bin orientation histograms per 16 px cell → normalised
  2×2-cell blocks (1,764): *which edge directions* appear where;
- **texture** – histogram of 8-neighbour Local Binary Patterns (256): fine surface patterns (petal veins, seed heads).

The ML models are tuned the same way as the CNNs (small grid, best validation macro F1).

In [ ]:
from src.baselines import run_baselines
if RETRAIN or not (out_2 / 'baselines.json').is_file():
    run_baselines(cfg)
b2 = json.loads((out_2 / 'baselines.json').read_text(encoding='utf-8'))
base = pd.DataFrame(b2['rows'])
family = lambda f, m: ('chance baseline' if 'dummy' in m else 'pretrained features + ML' if 'ImageNet' in f else 'classical CV + ML')
base['family'] = [family(f, m) for f, m in zip(base.features, base.model)]
base['label'] = [m + ('' if 'dummy' in m else ' | ' + ('ImageNet feat.' if 'ImageNet' in f else 'hand-crafted')) for f, m in zip(base.features, base.model)]
dl = exps.reset_index()[['experiment', 'test_accuracy', 'test_macro_f1', 'latency_ms', 'size_MB', 'train_minutes']]
dl = dl.assign(family='deep learning', label=dl.experiment, fit_seconds=dl.train_minutes * 60)
ladder = pd.concat([base[['family', 'label', 'test_accuracy', 'test_macro_f1', 'latency_ms', 'size_MB', 'fit_seconds']],
                    dl[['family', 'label', 'test_accuracy', 'test_macro_f1', 'latency_ms', 'size_MB', 'fit_seconds']]])
ladder = ladder.sort_values('test_macro_f1', ascending=False).reset_index(drop=True)
display(ladder.rename(columns={'fit_seconds': 'train_seconds'}))
from src.visualize import plot_method_ladder
fig = plot_method_ladder(ladder)
plt.show()
print('Baseline latency = feature extraction + classifier, one image already in memory (same rule as the CNN benchmark):',
      'hand-crafted features', b2['feature_latency_ms']['handcrafted'], 'ms, frozen ImageNet features', b2['feature_latency_ms']['deep_frozen'], 'ms.')

In [ ]:
best_of = lambda fam: ladder[ladder.family == fam].iloc[0]
chance, classic, probe, deep = (best_of(f) for f in ('chance baseline', 'classical CV + ML', 'pretrained features + ML', 'deep learning'))
cnn_best = ladder[ladder.label.str.startswith('cnn')].iloc[0]
display(Markdown(
    "**Conclusion (computed).**\n"
    f"- Chance level: {chance.label} scores macro F1 {chance.test_macro_f1:.3f} (accuracy {chance.test_accuracy:.3f}) – the floor.\n"
    f"- Best classical pipeline: **{classic.label}**, macro F1 {classic.test_macro_f1:.3f} "
    f"({classic.test_macro_f1 / max(chance.test_macro_f1, 1e-3):.1f}× chance). Hand-crafted colour/shape/texture features carry real information.\n"
    f"- Best from-scratch CNN: **{cnn_best.label}**, {cnn_best.test_macro_f1:.3f} ({cnn_best.test_macro_f1 - classic.test_macro_f1:+.3f} vs the best classical model). "
    + ("With ~2.5 k images, learning features from zero only matches a well-tuned classical pipeline.\n"
       if abs(cnn_best.test_macro_f1 - classic.test_macro_f1) < 0.05 else
       "Learning features end to end beats hand-designed ones.\n" if cnn_best.test_macro_f1 > classic.test_macro_f1 else
       "A small dataset is not enough for a CNN from scratch to beat the tuned classical pipeline.\n")
    + f"- Frozen ImageNet features + {probe.label.split(' |')[0]}: **{probe.test_macro_f1:.3f}** – no training on flowers at all, yet "
    f"{probe.test_macro_f1 - classic.test_macro_f1:+.3f} over the classical features: **the representation, not the classifier, is what matters**.\n"
    f"- Fine-tuning that representation ({deep.label}) adds a further {deep.test_macro_f1 - probe.test_macro_f1:+.3f}, reaching **{deep.test_macro_f1:.3f}**.\n"
    f"- Cost: the classical models train in seconds and are tiny; the fine-tuned network needs minutes of training and "
    f"{ladder.loc[ladder.label == deep.label, 'size_MB'].item():.0f} MB, but still answers in "
    f"{ladder.loc[ladder.label == deep.label, 'latency_ms'].item():.0f} ms on a CPU."))

#### What does the classical model rely on? (XAI for ML: group permutation importance)
For the best hand-crafted model, each feature family is shuffled across the validation images (colour of image A
paired with HOG of image B …) and the drop in macro F1 is measured. A large drop means the model depends on that family.

In [ ]:
imp = pd.DataFrame(b2['importance']).T.sort_values('f1_drop', ascending=False)
display(Markdown(f"Model explained: **{b2['importance_model']}** on hand-crafted features"), imp.round(4))
top = imp.index[0]
display(Markdown(
    f"**Conclusion (computed).** Shuffling **{top}** costs {imp.f1_drop.iloc[0]:.3f} macro F1, against "
    + ', '.join(f"{imp.f1_drop.iloc[i]:.3f} for {imp.index[i]}" for i in range(1, len(imp)))
    + ". The classical model is mostly a " + ('colour' if 'colour' in top else 'shape' if 'HOG' in top else 'texture')
    + " detector. That explains its ceiling: classes that share colours (red/pink roses vs tulips, yellow sunflowers vs "
    "dandelions) need petal shape and structure, which a fixed histogram cannot capture – the gap that learned features close."))

### [2.10] Explainable AI: why the deep models decide what they decide
A test score says *how often* a model is right, not *why*. Four checks back the conclusions with evidence:

1. **Grad-CAM** (Selvaraju et al., 2017): take the gradient of the predicted class score with respect to the last
   convolutional feature maps, average it per map to get a weight, and sum the weighted maps (ReLU) into a heatmap.
   Red = regions that pushed the score *up*. Written from scratch with forward/gradient hooks ([2.10a]).
2. **Deletion test** ([2.10b]): a heatmap can look convincing and still be wrong. Removing pixels in Grad-CAM order
   (most important first) must lower the class probability much faster than removing the same number of random
   pixels. The area under the curve (AUC) summarises it: **lower Grad-CAM AUC than random AUC = faithful explanation**.
3. **Calibration** ([2.10c]): when the model says 0.9, is it right 90 % of the time? Reliability diagram + expected
   calibration error (ECE, 0 = perfect). This decides whether the app may show the score as a confidence.
4. **Representation** ([2.10d]): t-SNE maps of the test images in each feature space and the silhouette score
   (+1 = tight, separate class clusters, 0 = mixed), from hand-crafted features to the fine-tuned network.

In [ ]:
from src.explain import run_explanations
if RETRAIN or not (out_2 / 'xai.json').is_file():
    run_explanations(cfg)
x2 = json.loads((out_2 / 'xai.json').read_text(encoding='utf-8'))
display(Image(out_2 / 'xai_gradcam_compare.png', width=900))

**How to read it.** Each row is one test image; the first column is the input, the others show where each model looks
(red = regions that raised the predicted class score).

**What the maps show.**
- The small CNNs' maps (7×7 grid) often *look* well placed on the flower. The fine-tuned MobileNetV3's maps are coarser
  (its last block is a 5×5 grid at 160 px) and sometimes spread to petal edges or the background.
- So looking at a map is not enough to judge it. That is exactly why the deletion test below measures whether the red
  regions really drive the prediction.
- The data can be wrong too: one "roses" test photo is a horse carriage scene. All models struggle with it except the
  fine-tuned one, which finds rose-like evidence in the decorations. Label noise like this sets a ceiling on any model.

In [ ]:
for name in x2['models']:
    if (out_2 / f'xai_mistakes_{name}.png').is_file() and name == m2['selected']:
        display(Image(out_2 / f'xai_mistakes_{name}.png', width=900))
display(Image(out_2 / 'xai_deletion.png'))
deletion = pd.DataFrame({name: {'AUC Grad-CAM order': v['deletion']['auc_gradcam'], 'AUC random order': v['deletion']['auc_random'],
                                'p(class) after removing 10 % (Grad-CAM)': v['deletion']['gradcam'][2],
                                'p(class) after removing 10 % (random)': v['deletion']['random'][2]}
                         for name, v in x2['models'].items()}).T.round(3)
display(deletion)

In [ ]:
lines = []
gaps = {}
for name, v in x2['models'].items():
    cam, rnd = np.array(v['deletion']['gradcam'][1:]), np.array(v['deletion']['random'][1:])
    gaps[name] = float(np.mean(rnd - cam))          # > 0: Grad-CAM pixels matter more than random pixels
    verdict = ('**faithful**' if gaps[name] > 0.03 else 'only weakly faithful' if gaps[name] > 0 else '**not faithful**')
    lines.append(f"- **{name}**: after removing 10 % of the pixels p = {v['deletion']['gradcam'][2]:.2f} (Grad-CAM order) vs "
                 f"{v['deletion']['random'][2]:.2f} (random); average gap {gaps[name]:+.3f} → the heatmap is {verdict}.")
passing = [n for n in sorted(gaps, key=gaps.get, reverse=True) if gaps[n] > 0.03]
failing = [n for n in gaps if gaps[n] <= 0.03]
display(Markdown('**Conclusion (computed).**\n' + '\n'.join(lines) + '\n\n'
    + (f"For {', '.join('**' + n + '**' for n in passing)} removing the highlighted region hurts clearly more than removing "
       "scattered pixels, so these heatmaps show evidence the model really uses: a *localised* region (the flower). "
       f"The gap is largest for **{passing[0]}** ({gaps[passing[0]]:+.3f}): it depends most on the flower itself. "
       if passing else "No model's heatmap beats random removal clearly. ")
    + (f"For {', '.join('**' + n + '**' for n in failing)} the heatmap is no better than random: scattered pixel removal hurts "
       "as much as removing the highlighted region, so the model uses colour and texture spread over the whole picture – "
       "the same strategy as the classical colour-histogram model. "
       if failing else "")
    + ("Augmentation is the most likely reason for the difference between the two small CNNs: random crops and flips push cnn_aug to find the "
       "flower wherever it is, instead of memorising global colour statistics. "
       if 'cnn_aug' in passing and 'cnn_no_aug' in failing else "")
    + "Caveat: random pixel removal is itself a strong perturbation (it adds noise), so the gap is a lower bound."))


In [ ]:
display(Image(out_2 / 'xai_calibration.png'))
ece = pd.Series({name: v['calibration']['ece'] for name, v in x2['models'].items()}, name='ECE').round(3)
display(ece.to_frame())
sel_ece = ece[m2['selected']]
bins = x2['models'][m2['selected']]['calibration']['bins']
direction = sum(b['count'] * (b['accuracy'] - b['confidence']) for b in bins) / sum(b['count'] for b in bins)
display(Markdown(
    f"**Conclusion (computed).** The selected model's ECE is {sel_ece:.3f}: its confidence is off by about "
    f"{sel_ece * 100:.1f} percentage points on average. "
    + ("That is small, so the app can show the score as a confidence and use a threshold (e.g. ask the user for a "
       "better photo below 0.6). " if sel_ece < 0.05 else
       "That is noticeable, so the app should label the number a score, not a probability, or calibrate it first "
       "(temperature scaling on the validation split). ")
    + (f"The error is **under-confidence** (accuracy is on average {direction:+.3f} above the stated confidence; e.g. "
       "predictions scored 0.9–1.0 are right almost every time). This is the expected effect of label smoothing (0.1) "
       "and the safe direction for an assistant: a high score can be trusted." if direction > 0 else
       f"The error is **over-confidence** (accuracy {direction:+.3f} below the stated confidence), so scores should be "
       "calibrated before they are shown.")))

In [ ]:
display(Image(out_2 / 'xai_tsne.png'))
spaces = {'hand-crafted features': x2.get('handcrafted'), 'ImageNet MobileNetV3 (frozen)': x2.get('imagenet_frozen'),
          **{f'{k} (trained)': v for k, v in x2['models'].items()}}
emb = pd.DataFrame({k: {'5-NN accuracy': v['knn_accuracy'], 'silhouette (cosine)': v['silhouette']}
                    for k, v in spaces.items() if v}).T.round(3)
display(emb)
order = emb['5-NN accuracy'].sort_values()
display(Markdown(
    "**Conclusion (computed).** In the hand-crafted space only "
    f"{emb.loc['hand-crafted features', '5-NN accuracy']:.0%} of the images have neighbours of the same class (chance = 20 %): "
    "visually similar images by colour/edge/texture statistics are often different flowers. "
    f"Frozen ImageNet features already reach {emb.loc['ImageNet MobileNetV3 (frozen)', '5-NN accuracy']:.0%}, the from-scratch CNNs "
    f"{emb.loc[[i for i in emb.index if i.startswith('cnn')], '5-NN accuracy'].max():.0%}, and the fine-tuned network "
    f"**{order.iloc[-1]:.0%}** ({order.index[-1]}), with the highest silhouette as well. The ranking matches the test scores "
    "of [2.9], so the accuracy gain comes from a genuinely better representation, not from luck on this test split. "
    "(Silhouette values look small because distances concentrate in high dimensions; compare them between rows, not to 1.)"))

### Task 2 conclusion – based on the findings above

In [ ]:
sel = m2['selected']
sel_row = exps.loc[sel]
cnn_best_name = max([n for n in exps.index if n.startswith('cnn')], key=lambda n: exps.loc[n, 'test_macro_f1'])
conf = np.array(m2['test_metrics'][sel]['confusion']); cl = m2['classes']
off = conf + conf.T; np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
n_err = conf.sum() - np.trace(conf)
sel_lat = ladder.loc[ladder.label == sel, 'latency_ms'].item()
classic_lat = ladder.loc[ladder.label == classic.label, 'latency_ms'].item()
display(Markdown(f'''
1. **Deep learning wins, and pretrained features are what make it win.** Test macro F1: chance
   {chance.test_macro_f1:.2f} → classical CV + ML {classic.test_macro_f1:.2f} → CNN from scratch {exps.loc[cnn_best_name, 'test_macro_f1']:.2f}
   → frozen ImageNet features + linear model {probe.test_macro_f1:.2f} → fine-tuned MobileNetV3 **{sel_row.test_macro_f1:.2f}**.
   With ~2.5 k training images a CNN trained from zero is only modestly better than a tuned classical pipeline;
   a representation learned on 1.2 M ImageNet photos does most of the work, fine-tuning adds the last
   {sel_row.test_macro_f1 - probe.test_macro_f1:+.2f}.
2. **Why (XAI):** the classical model is essentially a colour detector (shuffling the colour histogram costs
   {imp.f1_drop.iloc[0]:.2f} F1), so it confuses flowers that share colours. Deletion test (average gap, higher = relies on the highlighted
   region): {', '.join(f'{n} {g:+.3f}' for n, g in sorted(gaps.items(), key=lambda t: -t[1]))}. **{max(gaps, key=gaps.get)}** relies most on the
   localised flower region{'; ' + ', '.join(n for n in gaps if gaps[n] <= 0.03) + ' does not localise (no better than random)' if any(g <= 0.03 for g in gaps.values()) else ''}. The fine-tuned network's feature space puts
   {emb.loc[f'{sel} (trained)', '5-NN accuracy']:.0%} of the images next to images of the same class, against
   {emb.loc['hand-crafted features', '5-NN accuracy']:.0%} for hand-crafted features.
3. **Remaining errors:** {n_err} of {conf.sum()} test images; the most frequent confusion is **{cl[i]} ↔ {cl[j]}**
   ({off[i, j]} errors, {off[i, j] / max(n_err, 1):.0%} of all) – flowers with similar colours and cup-shaped petals.
4. **Choice for the app: `{sel}`** – best quality ({sel_row.test_accuracy:.1%} test accuracy), {sel_row.size_MB:.0f} MB,
   {sel_lat:.0f} ms per image on a CPU (the best classical pipeline needs {classic_lat:.0f} ms for {classic.test_macro_f1:.2f} F1).
   Its confidence is under-confident by about {ece[sel]:.2f} (ECE), so a high score can be trusted. If size were critical,
   the {exps.loc[cnn_best_name, 'size_MB']:.1f} MB CNN is the fallback, at a large cost in accuracy.
5. **Limits:** one public dataset of 5 classes in Flickr style; no "unknown" class; scores measured once on a
   {conf.sum()}-image test split (about ±2 % at 95 % confidence). Before real deployment: collect photos with the target
   camera, add an out-of-distribution class, and re-check calibration.
'''))

### Task 2 summary
| Rubric | Evidence in this notebook |
| --- | --- |
| 2.1 Data pipeline | manifest of the raw data, cleaning rules with a removal log (corrupt, small, odd shape, dHash duplicates, label conflicts), stratified split saved to CSV, shared preprocessing, train-only augmentation incl. noise injection |
| 2.2 Model | from-scratch CNN baseline and MobileNetV3 transfer learning (2-stage), learning-rate grid search on validation, best-epoch selection |
| 2.3 Evaluation | accuracy, macro F1, per-class precision/recall/F1, confusion matrices, failure gallery, size/latency/throughput/training time, saved model loaded and called; baselines from chance to classical ML; XAI (Grad-CAM + deletion test, calibration, embedding separation, permutation importance) |

## 3 Assistant — [3.1] run_mode

The same calls `app.py` makes, one per mode.

In [ ]:
# from src.assistant import MODES, run_mode
# list(MODES)

In [ ]:
# Colab only: save the new outputs to Google Drive before the runtime is deleted
if IN_COLAB:
    subprocess.run('zip -qr /content/drive/MyDrive/FCVG101-outputs.zip results artifacts data/splits exports', shell=True, check=True)
    print('Saved to My Drive/FCVG101-outputs.zip')